# CDCN trên OULU-NPU · Train/Dev mở rộng
Chọn A100 rồi chạy cell 1–4. Tập Train có 40 video, Dev có 30 video. Mỗi
video dùng 8 khung. Mẫu lấy phiên 1, điện thoại 1 với 5 loại video mỗi
người. Lượt train dùng batch 32, FP32 và 10 epoch.

Cell 3 dựng RGB, hộp mặt và PRNet trên CPU. Cell này lưu từng video đã
hoàn tất lên Drive và dùng lại sau khi Colab đổi runtime. Có thể chạy
cell 1–3 trên CPU rồi chuyển sang GPU và chạy lại cell 1–4. Cell 4 ghi
checkpoint, điểm và thời gian Dev.

In [ ]:
#@title 1. Kết nối Drive và chọn thư mục OULU-NPU
from pathlib import Path
import os
import shutil
import sys
import time
from google.colab import drive

drive.mount("/content/drive")
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
WORK = Path("/content/cdcn-prnet-expanded")
BASE = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-train-pilot-p1")
RUN = BASE / "expanded-runs" / str(time.time_ns())
for name, size in {"Protocols.tar": 417792, "Train_files.tar": 37753662976,
                   "Dev_files.tar": 28151992320}.items():
    source = DATA / name
    if not source.is_file() or source.stat().st_size != size:
        raise ValueError(f"Kiểm tra đường dẫn và dung lượng: {source}")
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)
import torch
print("GPU:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU", "| PyTorch:", torch.__version__)

In [ ]:
#@title 2. Nạp code đi kèm
BUNDLE_SHA256 = 'cfb8bab32e11915aae8f47339734af111bf0e73bf1d5253d590911c614c017ea'
BUNDLE = (
    '...'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged

def run(label, *args):
    run_logged(args, label=label, log=RUN / f"{label}-{time.time_ns()}.log",
               env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})


In [ ]:
#@title 3. Khôi phục hoặc dựng 560 khung MTCNN–PRNet
def prepare_prnet_environment():
    global PY, PRNET, RENDERER
    run("install-uv", sys.executable, "-m", "pip", "install", "uv")
    VENV = WORK / "venv"
    PY = VENV / "bin/python"
    if not PY.is_file():
        run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
    run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
        *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))
    PRNET = WORK / "PRNet"
    RENDERER = WORK / "PRNet-Depth-Generation"
    for path, repo, commit in [
        (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
        (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
    ]:
        if not path.exists():
            run(f"clone-{path.name}", "git", "clone", repo, path)
        run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
    WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
    def weight_matches(path):
        if not path.is_file():
            return False
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

    if not weight_matches(WEIGHT):
        partial = WEIGHT.with_suffix(".part")
        run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
            "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
        if not weight_matches(partial):
            raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
        partial.replace(WEIGHT)

import json
import zipfile

def digest(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

package = WORK / "scripts"
archive_binding = {}
for name in ("Train_files.tar", "Dev_files.tar"):
    path = DATA / name
    with path.open("rb") as stream:
        archive_binding[name] = {"bytes": path.stat().st_size,
                                 "prefix_sha256": hashlib.sha256(stream.read(65536)).hexdigest()}
config = {"protocol": digest(DATA / "Protocols.tar"), "archives": archive_binding,
          "scripts": {name: hashlib.sha256((package / name).read_text(encoding="utf-8").encode()).hexdigest()
                      for name in ("cdcn_prnet_smoke.py", "cdcn_prnet_expanded.py")}}
key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]
cache = BASE / "expanded-cache" / key
cache.mkdir(parents=True, exist_ok=True)
(RUN / "cache-config.json").write_text(json.dumps(config, indent=2))

pilot = BASE / "prepared-cache" / "c375c1dfc4bce1acc988"
metadata = pilot / "snapshot.json"
baseline = []
if metadata.is_file():
    meta = json.loads(metadata.read_text())
    if (meta["key"]["archives"] == archive_binding and meta["key"]["protocol"] == config["protocol"]
            and meta["key"]["preparer"] == config["scripts"]["cdcn_prnet_smoke.py"]):
        expected = "50ca8a42b3a5c3241714c181222b4b06e5098383b02b46a65071d5753be07443"
        if meta["sha256"] != expected:
            raise ValueError(f"Unexpected pilot snapshot: {metadata}")
        snapshot = pilot / f"assets-{expected}.zip"
        seed = (WORK / f"seed-{time.time_ns()}").resolve()
        with Progress("Restore the existing 48 frames"):
            local = seed.with_suffix(".zip")
            shutil.copyfile(snapshot, local)
            if digest(local) != expected:
                raise ValueError("Pilot ZIP checksum differs")
            with zipfile.ZipFile(local) as archive:
                if any(not (seed / name).resolve().is_relative_to(seed) for name in archive.namelist()):
                    raise ValueError("Invalid ZIP entry")
                archive.extractall(seed)
            local.unlink()
        baseline = ["--baseline", seed]

prepare_prnet_environment()
assets = WORK / f"expanded-{time.time_ns()}"
try:
    run("prepare-expanded", PY, "-u", package / "cdcn_prnet_expanded.py",
        "--data", DATA, "--cache", cache, "--output", assets, "--prnet", PRNET,
        "--renderer", RENDERER, *baseline)
finally:
    manifest = assets / "smoke_summary.json"
    if manifest.is_file():
        shutil.copyfile(manifest, RUN / "preparation.json")
print("EXPANDED_CACHE", cache)

In [ ]:
#@title 4. Train 10 epoch và chấm Dev
if not torch.cuda.is_available():
    raise RuntimeError("Chọn A100 và chạy lại cell 1–4 để khôi phục cache rồi train")
result = RUN / f"train-{time.time_ns()}"
run("train-expanded", sys.executable, "-u", WORK / "scripts/cdcn_prnet_train_pilot.py",
    "--assets", assets, "--output", result, "--epochs", 10, "--batch", 32,
    "--seed", 43, "--device", "cuda", "--profile", "expanded")
print((result / "pilot_result.json").read_text())
print("RUN_DIR", result)